In [5]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ── Ingest raw vendor feed ──────────────────────────────────────────
df = pd.read_csv('hfs_fund_data.csv')

print(f"✓ Ingested {len(df)} records from vendor feed")
print(f"✓ Ingestion timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✓ Columns             : {list(df.columns)}")
print(f"\nRaw data preview:")
df.head(20)

✓ Ingested 20 records from vendor feed
✓ Ingestion timestamp : 2026-03-23 12:35:47
✓ Columns             : ['fund_id', 'fund_name', 'strategy', 'aum_millions', 'monthly_return_pct', 'ytd_return_pct', 'net_exposure_pct', 'report_date', 'data_source', 'last_validated']

Raw data preview:


,fund_id,fund_name,strategy,aum_millions,monthly_return_pct,ytd_return_pct,net_exposure_pct,report_date,data_source,last_validated
0,HFS001,Apex Growth Fund,Long/Short Equity,542.3,3.2,8.7,45.2,2026-03-01,Bloomberg Feed,2026-03-22
1,HFS002,Summit Long Short,Long/Short Equity,318.7,-1.4,2.1,-32.8,2026-03-01,FactSet,2026-03-22
2,HFS003,NaN,Long/Short Equity,275.4,2.8,6.3,28.9,2026-03-01,Internal System,2026-03-22
3,HFS004,Ridgeline Capital,Long/Short Equity,487.1,0.9,4.2,51.3,2026-03-01,Fund Administrator,2026-03-22
4,HFS005,Atlas Global Macro,Global Macro,823.6,-2.1,1.8,-67.4,2026-03-01,Bloomberg Feed,2026-03-22
5,HFS006,Meridian Macro Fund,Global Macro,-200.0,1.3,5.6,42.1,2026-03-01,FactSet,2026-03-22
6,HFS007,Horizon Macro Partners,Global Macro,634.2,4.1,11.2,-55.8,2026-03-01,Prime Broker,2026-03-22
7,HFS008,Bridgepoint Merger Arb,Merger Arbitrage,291.8,1.1,3.4,18.6,2026-03-01,Fund Administrator,2026-03-22
8,HFS009,Nexus Event Driven,Merger Arbitrage,178.5,73.5,81.2,22.4,2026-03-01,Internal System,2026-03-22
9,HFS010,Pinnacle Arb Fund,Merger Arbitrage,445.9,0.8,2.9,15.3,2026-03-01,Bloomberg Feed,2026-03-22


In [6]:
issues = []

for idx, row in df.iterrows():
    row_issues = []

    # Check 1: Missing fund name
    if pd.isna(row['fund_name']) or str(row['fund_name']).strip() == '':
        row_issues.append('Missing fund name')

    # Check 2: AUM must be a positive number
    if pd.isna(row['aum_millions']) or not isinstance(row['aum_millions'], (int, float)) or row['aum_millions'] <= 0:
        row_issues.append('Invalid AUM')

    # Check 3: Return anomaly — flag anything beyond ±50%
    if pd.isna(row['monthly_return_pct']) or abs(row['monthly_return_pct']) > 50:
        row_issues.append(f"Return anomaly: {row['monthly_return_pct']}%")

    # Check 4: Stale data — report date older than 60 days
    try:
        report_dt = pd.to_datetime(row['report_date'])
        days_old = (datetime.now() - report_dt).days
        if days_old > 60:
            row_issues.append(f"Stale data: {days_old} days old")
    except:
        row_issues.append('Invalid report date')

    # Check 5: Net exposure breach — beyond ±150%
    if pd.isna(row['net_exposure_pct']) or abs(row['net_exposure_pct']) > 150:
        row_issues.append(f"Exposure breach: {row['net_exposure_pct']}%")

    if row_issues:
        issues.append({
            'fund_id'   : row['fund_id'],
            'fund_name' : row['fund_name'],
            'strategy'  : row['strategy'],
            'issues'    : '; '.join(row_issues)
        })

issues_df = pd.DataFrame(issues)

print("=" * 50)
print("  DATA QUALITY REPORT")
print("=" * 50)
print(f"  Total records checked  : {len(df)}")
print(f"  Records with issues    : {len(issues_df)}")
print(f"  Clean records          : {len(df) - len(issues_df)}")
print(f"  Data quality score     : {((len(df) - len(issues_df)) / len(df) * 100):.1f}%")
print("=" * 50)
print("\nFlagged records:")
issues_df

  DATA QUALITY REPORT
  Total records checked  : 20
  Records with issues    : 6
  Clean records          : 14
  Data quality score     : 70.0%

Flagged records:


,fund_id,fund_name,strategy,issues
0,HFS003,NaN,Long/Short Equity,Missing fund name
1,HFS006,Meridian Macro Fund,Global Macro,Invalid AUM
2,HFS009,Nexus Event Driven,Merger Arbitrage,Return anomaly: 73.5%
3,HFS012,Vantage Multi-Strategy,Multi-Strategy,Stale data: 101 days old
4,HFS015,Shoreline Credit Partners,Credit Long/Short,Exposure breach: 162.4%
5,HFS018,Sigma Quant Partners,Quant Equity,Invalid AUM


In [7]:
flagged_ids = issues_df['fund_id'].tolist()

# Split into clean and flagged
df_clean   = df[~df['fund_id'].isin(flagged_ids)].copy()
df_flagged = df[ df['fund_id'].isin(flagged_ids)].copy()

# Add quality flag column to full dataset
df['quality_flag'] = df['fund_id'].apply(
    lambda x: 'FLAGGED' if x in flagged_ids else 'CLEAN'
)

# Export all three outputs
df_clean.to_csv('hfs_clean_data.csv',     index=False)
df_flagged.to_csv('hfs_flagged_data.csv', index=False)
df.to_csv('hfs_validated_data.csv',       index=False)

print(f"✓ Clean records   : {len(df_clean)} → hfs_clean_data.csv")
print(f"✓ Flagged records : {len(df_flagged)} → hfs_flagged_data.csv")
print(f"✓ Full validated  : {len(df)} → hfs_validated_data.csv")
print(f"\nClean funds going to reporting:")
df_clean[['fund_id','fund_name','strategy','aum_millions','monthly_return_pct']].reset_index(drop=True)

✓ Clean records   : 14 → hfs_clean_data.csv
✓ Flagged records : 6 → hfs_flagged_data.csv
✓ Full validated  : 20 → hfs_validated_data.csv

Clean funds going to reporting:


,fund_id,fund_name,strategy,aum_millions,monthly_return_pct
0,HFS001,Apex Growth Fund,Long/Short Equity,542.3,3.2
1,HFS002,Summit Long Short,Long/Short Equity,318.7,-1.4
2,HFS004,Ridgeline Capital,Long/Short Equity,487.1,0.9
3,HFS005,Atlas Global Macro,Global Macro,823.6,-2.1
4,HFS007,Horizon Macro Partners,Global Macro,634.2,4.1
5,HFS008,Bridgepoint Merger Arb,Merger Arbitrage,291.8,1.1
6,HFS010,Pinnacle Arb Fund,Merger Arbitrage,445.9,0.8
7,HFS011,Keystone Multi-Strat,Multi-Strategy,712.4,2.4
8,HFS013,Cascade Multi-Strat,Multi-Strategy,521.3,-0.6
9,HFS014,Ironclad Credit Fund,Credit Long/Short,267.8,3.8


In [8]:
# ── Audit log — documents every run with full lineage ───────────────
audit_log = {
    'run_timestamp'      : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'source_file'        : 'hfs_fund_data.csv',
    'total_records'      : len(df),
    'clean_records'      : len(df_clean),
    'flagged_records'    : len(df_flagged),
    'quality_score_pct'  : round((len(df_clean) / len(df)) * 100, 1),
    'checks_applied'     : [
        'Check 1: Missing fund name',
        'Check 2: Invalid AUM (non-positive or missing)',
        'Check 3: Return anomaly (threshold: ±50%)',
        'Check 4: Stale data (threshold: >60 days)',
        'Check 5: Exposure breach (threshold: ±150%)'
    ],
    'flagged_records_detail' : issues_df.to_dict(orient='records'),
    'output_files'       : [
        'hfs_clean_data.csv',
        'hfs_flagged_data.csv',
        'hfs_validated_data.csv',
        'hfs_strategy_summary.csv'
    ],
    'pipeline_version'   : '1.0',
    'analyst'            : 'Sakshi Aade'
}

# Write audit log to JSON
with open('audit_log.json', 'w') as f:
    json.dump(audit_log, f, indent=2)

print("✓ Audit log written to audit_log.json")
print("\nAudit Summary:")
print(f"  Run timestamp    : {audit_log['run_timestamp']}")
print(f"  Source file      : {audit_log['source_file']}")
print(f"  Quality score    : {audit_log['quality_score_pct']}%")
print(f"  Pipeline version : {audit_log['pipeline_version']}")
print(f"  Analyst          : {audit_log['analyst']}")
print("\nFull audit log:")
print(json.dumps(audit_log, indent=2))

✓ Audit log written to audit_log.json

Audit Summary:
  Run timestamp    : 2026-03-23 12:37:43
  Source file      : hfs_fund_data.csv
  Quality score    : 70.0%
  Pipeline version : 1.0
  Analyst          : Sakshi Aade

Full audit log:
{
  "run_timestamp": "2026-03-23 12:37:43",
  "source_file": "hfs_fund_data.csv",
  "total_records": 20,
  "clean_records": 14,
  "flagged_records": 6,
  "quality_score_pct": 70.0,
  "checks_applied": [
    "Check 1: Missing fund name",
    "Check 2: Invalid AUM (non-positive or missing)",
    "Check 3: Return anomaly (threshold: \u00b150%)",
    "Check 4: Stale data (threshold: >60 days)",
    "Check 5: Exposure breach (threshold: \u00b1150%)"
  ],
  "flagged_records_detail": [
    {
      "fund_id": "HFS003",
      "fund_name": NaN,
      "strategy": "Long/Short Equity",
      "issues": "Missing fund name"
    },
    {
      "fund_id": "HFS006",
      "fund_name": "Meridian Macro Fund",
      "strategy": "Global Macro",
      "issues": "Invalid AUM"
  

In [9]:
# ── Performance KPIs on clean data only ─────────────────────────────
summary = df_clean.groupby('strategy').agg(
    fund_count        = ('fund_id',            'count'),
    total_aum_mm      = ('aum_millions',        'sum'),
    avg_monthly_ret   = ('monthly_return_pct',  'mean'),
    avg_ytd_ret       = ('ytd_return_pct',       'mean'),
    avg_net_exposure  = ('net_exposure_pct',     'mean')
).round(2).reset_index()

# Export for Tableau
summary.to_csv('hfs_strategy_summary.csv', index=False)

print("=" * 60)
print("  HFS PERFORMANCE SUMMARY — CLEAN DATA ONLY")
print("=" * 60)
print(f"  Report date : {datetime.now().strftime('%B %Y')}")
print(f"  Based on    : {len(df_clean)} clean funds ({len(df)} total ingested)")
print("=" * 60)
print()
print(summary.to_string(index=False))
print()
print(f"✓ Strategy summary exported to hfs_strategy_summary.csv")
print(f"\nTotal AUM across all clean funds: ${summary['total_aum_mm'].sum():,.1f}M")
print(f"Best performing strategy: {summary.loc[summary['avg_monthly_ret'].idxmax(), 'strategy']} ({summary['avg_monthly_ret'].max():.2f}%)")
print(f"Largest strategy by AUM : {summary.loc[summary['total_aum_mm'].idxmax(), 'strategy']} (${summary['total_aum_mm'].max():,.1f}M)")

  HFS PERFORMANCE SUMMARY — CLEAN DATA ONLY
  Report date : March 2026
  Based on    : 14 clean funds (20 total ingested)

         strategy  fund_count  total_aum_mm  avg_monthly_ret  avg_ytd_ret  avg_net_exposure
Credit Long/Short           2         466.2             0.95         4.85              6.75
     Global Macro           2        1457.8             1.00         6.50            -61.60
Long/Short Equity           3        1348.1             0.90         5.00             21.23
 Merger Arbitrage           2         737.7             0.95         3.15             16.95
   Multi-Strategy           2        1233.7             0.90         4.50            -34.15
     Quant Equity           3        1963.3             1.33         5.33             15.73

✓ Strategy summary exported to hfs_strategy_summary.csv

Total AUM across all clean funds: $7,206.8M
Best performing strategy: Quant Equity (1.33%)
Largest strategy by AUM : Quant Equity ($1,963.3M)


In [ ]:
import sqlite3

# ── Load CSVs into an in-memory SQLite database ──────────────────────
conn = sqlite3.connect(':memory:')
df_clean.to_sql('hfs_clean_data',     conn, index=False, if_exists='replace')
df_flagged.to_sql('hfs_flagged_data', conn, index=False, if_exists='replace')
df.to_sql('hfs_validated_data',       conn, index=False, if_exists='replace')
print("✓ Tables loaded into SQLite: hfs_clean_data, hfs_flagged_data, hfs_validated_data")

# ── Helper to run and display any query cleanly ──────────────────────
def run_query(title, sql):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")
    result = pd.read_sql_query(sql, conn)
    print(result.to_string(index=False))
    return result

# ── Query 1: AUM and returns by strategy ────────────────────────────
run_query("Q1: Total AUM & Avg Return by Strategy", """
    SELECT 
        strategy,
        COUNT(*)                          AS fund_count,
        ROUND(SUM(aum_millions), 1)       AS total_aum_millions,
        ROUND(AVG(monthly_return_pct), 2) AS avg_monthly_return_pct
    FROM hfs_clean_data
    GROUP BY strategy
    ORDER BY total_aum_millions DESC
""")

# ── Query 2: Top 5 performing funds this month ───────────────────────
run_query("Q2: Top 5 Performing Funds — Current Month", """
    SELECT 
        fund_name,
        strategy,
        aum_millions,
        monthly_return_pct,
        ytd_return_pct
    FROM hfs_clean_data
    WHERE monthly_return_pct > 0
    ORDER BY monthly_return_pct DESC
    LIMIT 5
""")


✓ Tables loaded into SQLite: hfs_clean_data, hfs_flagged_data, hfs_validated_data

  Q1: Total AUM & Avg Return by Strategy
         strategy  fund_count  total_aum_millions  avg_monthly_return_pct
     Quant Equity           3              1963.3                    1.33
     Global Macro           2              1457.8                    1.00
Long/Short Equity           3              1348.1                    0.90
   Multi-Strategy           2              1233.7                    0.90
 Merger Arbitrage           2               737.7                    0.95
Credit Long/Short           2               466.2                    0.95

  Q2: Top 5 Performing Funds — Current Month
             fund_name          strategy  aum_millions  monthly_return_pct  ytd_return_pct
    Quantum Systematic      Quant Equity         876.5                 5.2            14.3
Horizon Macro Partners      Global Macro         634.2                 4.1            11.2
  Ironclad Credit Fund Credit Long/Shor

,fund_name,strategy,aum_millions,monthly_return_pct,ytd_return_pct
0,Quantum Systematic,Quant Equity,876.5,5.2,14.3
1,Horizon Macro Partners,Global Macro,634.2,4.1,11.2
2,Ironclad Credit Fund,Credit Long/Short,267.8,3.8,9.4
3,Apex Growth Fund,Long/Short Equity,542.3,3.2,8.7
4,Keystone Multi-Strat,Multi-Strategy,712.4,2.4,7.8


In [13]:
# ── Query 3: Exposure monitoring ─────────────────────────────────────
run_query("Q3: Exposure Monitoring — Risk Flags", """
    SELECT
        fund_name,
        strategy,
        net_exposure_pct,
        CASE 
            WHEN ABS(net_exposure_pct) > 100 THEN 'HIGH RISK'
            WHEN ABS(net_exposure_pct) > 75  THEN 'ELEVATED'
            ELSE 'NORMAL'
        END AS exposure_flag
    FROM hfs_clean_data
    ORDER BY ABS(net_exposure_pct) DESC
""")


  Q3: Exposure Monitoring — Risk Flags
              fund_name          strategy  net_exposure_pct exposure_flag
     Quantum Systematic      Quant Equity              71.8        NORMAL
     Atlas Global Macro      Global Macro             -67.4        NORMAL
Vector Algorithmic Fund      Quant Equity             -63.5        NORMAL
   Ironclad Credit Fund Credit Long/Short              62.1        NORMAL
 Horizon Macro Partners      Global Macro             -55.8        NORMAL
      Ridgeline Capital Long/Short Equity              51.3        NORMAL
   Westfield Credit L/S Credit Long/Short             -48.6        NORMAL
       Apex Growth Fund Long/Short Equity              45.2        NORMAL
   Keystone Multi-Strat    Multi-Strategy             -38.9        NORMAL
     Helix Quant Equity      Quant Equity              38.9        NORMAL
      Summit Long Short Long/Short Equity             -32.8        NORMAL
    Cascade Multi-Strat    Multi-Strategy             -29.4        NORMA

,fund_name,strategy,net_exposure_pct,exposure_flag
0,Quantum Systematic,Quant Equity,71.8,NORMAL
1,Atlas Global Macro,Global Macro,-67.4,NORMAL
2,Vector Algorithmic Fund,Quant Equity,-63.5,NORMAL
3,Ironclad Credit Fund,Credit Long/Short,62.1,NORMAL
4,Horizon Macro Partners,Global Macro,-55.8,NORMAL
5,Ridgeline Capital,Long/Short Equity,51.3,NORMAL
6,Westfield Credit L/S,Credit Long/Short,-48.6,NORMAL
7,Apex Growth Fund,Long/Short Equity,45.2,NORMAL
8,Keystone Multi-Strat,Multi-Strategy,-38.9,NORMAL
9,Helix Quant Equity,Quant Equity,38.9,NORMAL


In [14]:
# ── Query 4: Data timeliness check ───────────────────────────────────
run_query("Q4: Data Timeliness — Days Since Last Report", """
    SELECT
        fund_name,
        strategy,
        report_date,
        CAST(JULIANDAY('now') - JULIANDAY(report_date) AS INTEGER) AS days_since_report,
        CASE
            WHEN JULIANDAY('now') - JULIANDAY(report_date) > 60 THEN 'STALE'
            WHEN JULIANDAY('now') - JULIANDAY(report_date) > 30 THEN 'AGING'
            ELSE 'CURRENT'
        END AS timeliness_status
    FROM hfs_clean_data
    ORDER BY days_since_report DESC
""")


  Q4: Data Timeliness — Days Since Last Report
              fund_name          strategy report_date  days_since_report timeliness_status
       Apex Growth Fund Long/Short Equity  2026-03-01                 22           CURRENT
      Summit Long Short Long/Short Equity  2026-03-01                 22           CURRENT
      Ridgeline Capital Long/Short Equity  2026-03-01                 22           CURRENT
     Atlas Global Macro      Global Macro  2026-03-01                 22           CURRENT
 Horizon Macro Partners      Global Macro  2026-03-01                 22           CURRENT
 Bridgepoint Merger Arb  Merger Arbitrage  2026-03-01                 22           CURRENT
      Pinnacle Arb Fund  Merger Arbitrage  2026-03-01                 22           CURRENT
   Keystone Multi-Strat    Multi-Strategy  2026-03-01                 22           CURRENT
    Cascade Multi-Strat    Multi-Strategy  2026-03-01                 22           CURRENT
   Ironclad Credit Fund Credit Long/Short 

,fund_name,strategy,report_date,days_since_report,timeliness_status
0,Apex Growth Fund,Long/Short Equity,2026-03-01,22,CURRENT
1,Summit Long Short,Long/Short Equity,2026-03-01,22,CURRENT
2,Ridgeline Capital,Long/Short Equity,2026-03-01,22,CURRENT
3,Atlas Global Macro,Global Macro,2026-03-01,22,CURRENT
4,Horizon Macro Partners,Global Macro,2026-03-01,22,CURRENT
5,Bridgepoint Merger Arb,Merger Arbitrage,2026-03-01,22,CURRENT
6,Pinnacle Arb Fund,Merger Arbitrage,2026-03-01,22,CURRENT
7,Keystone Multi-Strat,Multi-Strategy,2026-03-01,22,CURRENT
8,Cascade Multi-Strat,Multi-Strategy,2026-03-01,22,CURRENT
9,Ironclad Credit Fund,Credit Long/Short,2026-03-01,22,CURRENT


In [15]:
# ── Query 5: Quality summary across full dataset ──────────────────────
run_query("Q5: Data Quality Summary — Clean vs Flagged", """
    SELECT
        quality_flag,
        COUNT(*)                                        AS record_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_of_total
    FROM hfs_validated_data
    GROUP BY quality_flag
""")


  Q5: Data Quality Summary — Clean vs Flagged
quality_flag  record_count  pct_of_total
       CLEAN            14          70.0
     FLAGGED             6          30.0


,quality_flag,record_count,pct_of_total
0,CLEAN,14,70.0
1,FLAGGED,6,30.0
